In [9]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
monmlp-like regression pipeline in Python (MLPRegressor)
-------------------------------------------------------
- Mirrors the R caret pipeline logic as much as possible.
- 12 files × 4 targets (Jsc, Voc, FF, PCEmax)
- Feature selection:
    * numeric columns only
    * drop rows with any NA
    * drop the four targets (Jsc, Voc, FF, PCEmax) from features
    * drop zero-variance features
    * drop near-duplicate highly correlated features (|r| > 0.99999)
- Splits:
    * OOD = farthest 10% samples from centroid in feature space
    * Train = remaining 90%
    * CV10_OOF = 10-fold CV (OOF) on Train with best hyperparameters
- Scaling:
    * MinMaxScaler(-1, 1) fit on Train, applied to Train & OOD
- Model:
    * MLPRegressor (sklearn) as "monmlp" substitute
    * Hyperparameter tuning via GridSearchCV on Train
- Outputs:
    * Model bundle .pkl for each (file, target)
    * Prediction CSV for each (file, target), with rows from:
        - Train
        - CV10_OOF
        - OOD_Test
      Columns: SampleID, Observed, Predicted, Type
    * Summary CSV (fixed_YYYYMMDD_monmlp_summary.csv)
        - Train / CV10_OOF / OOD metrics (R2, RMSE, MAE, RPD)
        - Overfit_Flag, Overfit_Label
        - Delta_R2_TrainMinusCV10
        - Delta_RMSE_CV10MinusTrain
    * Importance CSV (permutation importance on Train, baseline = Train_R2)
"""

import os
import pickle
from datetime import datetime

import numpy as np
import pandas as pd
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


# -------------------------------
# USER SETTINGS
# -------------------------------

BASE_PATH = "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/rebuilt_merged_data/"

TARGET_VARS = ["Jsc", "Voc", "FF", "PCEmax"]
TODAY = datetime.today().strftime("%Y%m%d")

FILE_NAMES = [
    "n_base.csv", "n_base_OH_rebuilt.csv", "n_base_FP_rebuilt.csv",
    "n_all.csv",  "n_all_OH_rebuilt.csv",  "n_all_FP_rebuilt.csv",
    "m_base.csv", "m_base_OH_rebuilt.csv", "m_base_FP_rebuilt.csv",
    "m_all.csv",  "m_all_OH_rebuilt.csv",  "m_all_FP_rebuilt.csv"
]

MODEL_NAME = "monmlp"  # for filenames
OOD_RATIO = 0.10

# Overfit rule thresholds (same idea as R)
OVERFIT_DELTA_R2_THR = 0.10
OVERFIT_DELTA_RMSE_REL = 0.10  # 10% of Train_RMSE


# -------------------------------
# Helper functions
# -------------------------------

def safe_r2(y_true, y_pred):
    if y_pred is None or len(y_pred) == 0:
        return np.nan
    try:
        return float(r2_score(y_true, y_pred))
    except Exception:
        return np.nan


def safe_rmse(y_true, y_pred):
    if y_pred is None or len(y_pred) == 0:
        return np.nan
    try:
        return float(np.sqrt(mean_squared_error(y_true, y_pred)))
    except Exception:
        return np.nan


def safe_mae(y_true, y_pred):
    if y_pred is None or len(y_pred) == 0:
        return np.nan
    try:
        return float(mean_absolute_error(y_true, y_pred))
    except Exception:
        return np.nan


def safe_rpd(y_true, rmse_val):
    if rmse_val is None or not np.isfinite(rmse_val) or rmse_val == 0:
        return np.nan
    try:
        return float(np.std(y_true, ddof=1) / rmse_val)
    except Exception:
        return np.nan


def create_ood_split(df_numeric, feature_cols, ood_ratio=0.1):
    """
    Farthest 10% from centroid in feature space = OOD.
    Rest = Train.
    """
    X = df_numeric[feature_cols].to_numpy()
    centroid = X.mean(axis=0)
    dists = np.sqrt(((X - centroid) ** 2).sum(axis=1))
    n_samples = X.shape[0]
    n_ood = max(1, int(np.floor(n_samples * ood_ratio)))
    ood_idx = np.argsort(-dists)[:n_ood]  # farthest
    mask = np.ones(n_samples, dtype=bool)
    mask[ood_idx] = False
    train_idx = np.where(mask)[0]
    return train_idx, ood_idx


def scale_neg1_to_1(train_df, test_df, feature_cols):
    scaler = MinMaxScaler(feature_range=(-1, 1))
    scaler.fit(train_df[feature_cols].to_numpy())

    train_scaled = train_df.copy()
    test_scaled = test_df.copy()
    train_scaled[feature_cols] = scaler.transform(train_df[feature_cols].to_numpy())
    test_scaled[feature_cols] = scaler.transform(test_df[feature_cols].to_numpy())
    return train_scaled, test_scaled, scaler


def remove_zero_variance(df, feature_cols):
    vars_ = df[feature_cols].var(axis=0, ddof=1)
    kept = [c for c in feature_cols if np.isfinite(vars_[c]) and vars_[c] > 0]
    return kept


def remove_high_corr_features(df, feature_cols, cutoff=0.99999):
    """
    Rough equivalent of caret::findCorrelation with a very high cutoff.
    We drop the later feature in each highly correlated pair (upper-triangular).
    """
    if len(feature_cols) <= 1:
        return feature_cols

    X = df[feature_cols].to_numpy()
    corr = np.corrcoef(X, rowvar=False)
    to_drop = set()
    n = len(feature_cols)

    for i in range(n):
        for j in range(i + 1, n):
            if j in to_drop:
                continue
            r = corr[i, j]
            if np.isfinite(r) and abs(r) > cutoff:
                # drop j (similar to caret behavior)
                to_drop.add(j)

    kept = [feature_cols[i] for i in range(n) if i not in to_drop]
    return kept


def tune_monmlp(X_train, y_train, random_state=42):
    """
    Hyperparameter tuning for MLPRegressor using RMSE (neg_root_mean_squared_error).
    Returns: best_estimator_, best_params_
    """
    base = MLPRegressor(
        max_iter=2000,
        random_state=random_state
    )
    param_grid = {
        "hidden_layer_sizes": [(32,), (64,), (128,)],
        "alpha": [1e-4, 1e-3, 1e-2]
    }
    gcv = GridSearchCV(
        estimator=base,
        param_grid=param_grid,
        scoring="neg_root_mean_squared_error",
        cv=10,
        n_jobs=-1
    )
    gcv.fit(X_train, y_train)
    return gcv.best_estimator_, gcv.best_params_


def get_oof_predictions(best_params, X_train, y_train, random_state=42, n_splits=10):
    """
    10-fold OOF predictions using fixed best hyperparameters.
    """
    n = X_train.shape[0]
    oof_pred = np.full(n, np.nan, dtype=float)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    for train_idx, val_idx in kf.split(X_train):
        model = MLPRegressor(
            max_iter=2000,
            random_state=random_state,
            hidden_layer_sizes=best_params["hidden_layer_sizes"],
            alpha=best_params["alpha"]
        )
        model.fit(X_train[train_idx], y_train[train_idx])
        oof_pred[val_idx] = model.predict(X_train[val_idx])

    return oof_pred


def calc_permutation_importance(model, df_train_scaled, target_col, feature_cols, base_r2, random_state=42):
    """
    Very simple permutation importance on Train:
    importance = (Train_R2 baseline) - (R2 after shuffling 1 feature).
    """
    rng = np.random.default_rng(random_state)
    y_true = df_train_scaled[target_col].to_numpy()
    X_orig = df_train_scaled[feature_cols].to_numpy()

    importances = {}
    for i, col in enumerate(feature_cols):
        X_perm = X_orig.copy()
        # permute only this column
        X_perm[:, i] = rng.permutation(X_perm[:, i])
        y_pred_perm = model.predict(X_perm)
        new_r2 = safe_r2(y_true, y_pred_perm)
        if not np.isfinite(new_r2):
            new_r2 = 0.0
        importances[col] = float(base_r2 - new_r2)
    return importances


# -------------------------------
# Main runner
# -------------------------------

def run_monmlp_pipeline():
    out_dir = os.path.join(
        os.getcwd(),
        f"{TODAY}_Rebuilt_12Files_{MODEL_NAME}_Train_CV10OOF_OOD"
    )
    os.makedirs(out_dir, exist_ok=True)

    print("=" * 60)
    print(f"--- monmlp (MLPRegressor) Analysis Started: {TODAY} ---")
    print(f"[INFO] Input base path: {BASE_PATH}")
    print(f"[INFO] Output dir     : {out_dir}")
    print("=" * 60)

    summary_rows = []
    importance_rows = []

    for file_name in FILE_NAMES:
        file_path = os.path.join(BASE_PATH, file_name)
        if not os.path.isfile(file_path):
            print(f"Skipping: {file_name} (not found)")
            continue

        print(f"\n=== Processing file: {file_name} ===")

        try:
            df_raw = pd.read_csv(file_path, index_col=0)
        except Exception as e:
            print(f"  [ERROR] Failed to read file: {e}")
            continue

        # Drop 'X' col if present (same as R)
        if "X" in df_raw.columns:
            df_raw = df_raw.drop(columns=["X"])

        # numeric columns only
        df_numeric = df_raw.select_dtypes(include=[np.number]).dropna()
        if df_numeric.shape[0] < 20:
            print(f"  [WARN] Too few complete rows (<20). Skipped.")
            continue

        for target in TARGET_VARS:
            if target not in df_numeric.columns:
                continue

            print(f"  --- Target: {target} ---")

            # ------------------------------
            # Feature selection
            # ------------------------------
            # 4 targets are all excluded from features
            feature_cols = [c for c in df_numeric.columns if c not in TARGET_VARS]
            if len(feature_cols) == 0:
                print("    [WARN] No features after excluding targets. Skipped.")
                continue

            # zero variance
            feature_cols = remove_zero_variance(df_numeric, feature_cols)
            if len(feature_cols) == 0:
                print("    [WARN] No features after zero-variance removal. Skipped.")
                continue

            # near-duplicate correlation
            feature_cols = remove_high_corr_features(df_numeric, feature_cols, cutoff=0.99999)
            if len(feature_cols) == 0:
                print("    [WARN] No features after high-corr removal. Skipped.")
                continue

            # ------------------------------
            # OOD split
            # ------------------------------
            train_idx, ood_idx = create_ood_split(df_numeric, feature_cols, ood_ratio=OOD_RATIO)

            df_train = df_numeric.iloc[train_idx].copy()
            df_ood = df_numeric.iloc[ood_idx].copy()

            # keep sample IDs as strings
            sample_ids_train = df_train.index.astype(str).tolist()
            sample_ids_ood = df_ood.index.astype(str).tolist()

            # ------------------------------
            # Scaling -1..1 (fit on Train)
            # ------------------------------
            df_train_scaled, df_ood_scaled, scaler = scale_neg1_to_1(df_train, df_ood, feature_cols)

            X_train = df_train_scaled[feature_cols].to_numpy()
            y_train = df_train_scaled[target].to_numpy()
            X_ood = df_ood_scaled[feature_cols].to_numpy()
            y_ood = df_ood_scaled[target].to_numpy()

            # ------------------------------
            # Hyperparameter tuning (GridSearchCV)
            # ------------------------------
            try:
                best_model, best_params = tune_monmlp(X_train, y_train, random_state=42)
            except Exception as e:
                print(f"    [ERROR] Hyperparameter tuning failed for target={target}: {e}")
                continue

            # ------------------------------
            # Train final model on full Train
            # ------------------------------
            best_model.fit(X_train, y_train)

            # Train predictions
            y_train_pred = best_model.predict(X_train)

            # OOD predictions
            y_ood_pred = best_model.predict(X_ood)

            # ------------------------------
            # CV10 OOF predictions (using best hyperparams)
            # ------------------------------
            y_oof_pred = get_oof_predictions(best_params, X_train, y_train, random_state=42, n_splits=10)

            # ------------------------------
            # Metrics
            # ------------------------------
            # Train
            train_r2 = safe_r2(y_train, y_train_pred)
            train_rmse = safe_rmse(y_train, y_train_pred)
            train_mae = safe_mae(y_train, y_train_pred)
            train_rpd = safe_rpd(y_train, train_rmse)

            # CV10_OOF
            cv10_r2 = safe_r2(y_train, y_oof_pred)
            cv10_rmse = safe_rmse(y_train, y_oof_pred)
            cv10_mae = safe_mae(y_train, y_oof_pred)
            cv10_rpd = safe_rpd(y_train, cv10_rmse)

            # OOD
            ood_r2 = safe_r2(y_ood, y_ood_pred)
            ood_rmse = safe_rmse(y_ood, y_ood_pred)
            ood_mae = safe_mae(y_ood, y_ood_pred)
            ood_rpd = safe_rpd(y_ood, ood_rmse)

            # Overfit check (Train vs CV10_OOF)
            delta_r2 = train_r2 - cv10_r2
            delta_rmse = cv10_rmse - train_rmse

            overfit_flag = (
                (np.isfinite(delta_r2) and delta_r2 >= OVERFIT_DELTA_R2_THR) or
                (np.isfinite(delta_rmse) and np.isfinite(train_rmse) and
                 delta_rmse >= OVERFIT_DELTA_RMSE_REL * abs(train_rmse))
            )
            overfit_label = "Overfit_suspected" if overfit_flag else "OK"

            # ------------------------------
            # Save model bundle (pickle)
            # ------------------------------
            file_stem = os.path.splitext(file_name)[0]
            model_file = os.path.join(
                out_dir,
                f"fixed_{TODAY}_{MODEL_NAME}_model_{file_stem}_{target}.pkl"
            )
            model_bundle = {
                "model": best_model,
                "scaler": scaler,
                "features": feature_cols,
                "target": target,
                "train_index": sample_ids_train,
                "ood_index": sample_ids_ood,
                "best_params": best_params
            }
            with open(model_file, "wb") as f:
                pickle.dump(model_bundle, f)

            # ------------------------------
            # Prediction CSV (Train / CV10_OOF / OOD_Test)
            # Columns: SampleID, Observed, Predicted, Type
            # ------------------------------
            df_train_pred = pd.DataFrame({
                "SampleID": sample_ids_train,
                "Observed": y_train,
                "Predicted": y_train_pred,
                "Type": "Train"
            })

            df_oof_pred = pd.DataFrame({
                "SampleID": sample_ids_train,
                "Observed": y_train,
                "Predicted": y_oof_pred,
                "Type": "CV10_OOF"
            })

            df_ood_pred = pd.DataFrame({
                "SampleID": sample_ids_ood,
                "Observed": y_ood,
                "Predicted": y_ood_pred,
                "Type": "OOD_Test"
            })

            df_pred_all = pd.concat([df_train_pred, df_oof_pred, df_ood_pred], axis=0, ignore_index=True)

            pred_file = os.path.join(
                out_dir,
                f"fixed_{TODAY}_{MODEL_NAME}_{file_stem}_{target}_pred.csv"
            )
            df_pred_all.to_csv(pred_file, index=False)

            # ------------------------------
            # Permutation Importance on Train
            # ------------------------------
            importances = calc_permutation_importance(
                best_model,
                df_train_scaled,
                target_col=target,
                feature_cols=feature_cols,
                base_r2=train_r2,
                random_state=42
            )

            for feat, imp in importances.items():
                if not np.isfinite(imp) or abs(imp) <= 1e-6:
                    continue
                importance_rows.append({
                    "File": file_name,
                    "Target": target,
                    "Feature": feat,
                    "Importance_Mean": imp,
                    "Importance_Type": "Permutation_R2drop"
                })

            # ------------------------------
            # Summary row
            # ------------------------------
            summary_rows.append({
                "File": file_name,
                "Target": target,
                "Train_R2": train_r2,
                "Train_RMSE": train_rmse,
                "Train_MAE": train_mae,
                "Train_RPD": train_rpd,
                "CV10_R2": cv10_r2,
                "CV10_RMSE": cv10_rmse,
                "CV10_MAE": cv10_mae,
                "CV10_RPD": cv10_rpd,
                "OOD_R2": ood_r2,
                "OOD_RMSE": ood_rmse,
                "OOD_MAE": ood_mae,
                "OOD_RPD": ood_rpd,
                "n_samples": df_numeric.shape[0],
                "n_features": len(feature_cols),
                "Best_Params": f"hidden_layer_sizes={best_params['hidden_layer_sizes']}, alpha={best_params['alpha']}",
                "Overfit_Flag": overfit_flag,
                "Overfit_Label": overfit_label,
                "Delta_R2_TrainMinusCV10": delta_r2,
                "Delta_RMSE_CV10MinusTrain": delta_rmse
            })

            print(
                f"    Target={target} | TrainR2={train_r2:.3f} | "
                f"CV10R2={cv10_r2:.3f} | OODR2={ood_r2:.3f} | {overfit_label}"
            )

    # ------------------------------
    # Save summary & importance & overfit judgement
    # ------------------------------
    summary_df = pd.DataFrame(summary_rows)
    importance_df = pd.DataFrame(importance_rows)

    summary_file = os.path.join(out_dir, f"fixed_{TODAY}_{MODEL_NAME}_summary.csv")
    importance_file = os.path.join(out_dir, f"fixed_{TODAY}_{MODEL_NAME}_feature_importance.csv")
    overfit_file = os.path.join(out_dir, f"fixed_{TODAY}_{MODEL_NAME}_overfit_judgement.csv")

    summary_df.to_csv(summary_file, index=False)
    importance_df.to_csv(importance_file, index=False)

    # overfit judgement table: select columns
    overfit_cols = [
        "File", "Target",
        "Train_R2", "CV10_R2", "OOD_R2",
        "Train_RMSE", "CV10_RMSE", "OOD_RMSE",
        "Overfit_Flag", "Overfit_Label",
        "Delta_R2_TrainMinusCV10",
        "Delta_RMSE_CV10MinusTrain",
        "Best_Params", "n_samples", "n_features"
    ]
    overfit_df = summary_df[overfit_cols]
    overfit_df.to_csv(overfit_file, index=False)

    print("=" * 60)
    print("monmlp pipeline finished.")
    print(f"Summary    : {summary_file}")
    print(f"Importance : {importance_file}")
    print(f"Overfit    : {overfit_file}")
    print("=" * 60)


if __name__ == "__main__":
    run_monmlp_pipeline()


--- monmlp (MLPRegressor) Analysis Started: 20251218 ---
[INFO] Input base path: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/rebuilt_merged_data/
[INFO] Output dir     : /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/Cal _regression/monmlp/20251218_Rebuilt_12Files_monmlp_Train_CV10OOF_OOD

=== Processing file: n_base.csv ===
  --- Target: Jsc ---


/home/is/i-yasuaki/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/is/i-yasuaki/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/is/i-yasuaki/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/is/i-yasuaki/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
/hom

/home/is/i-yasuaki/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/is/i-yasuaki/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/is/i-yasuaki/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/is/i-yasuaki/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
/hom

/home/is/i-yasuaki/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/is/i-yasuaki/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/is/i-yasuaki/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/is/i-yasuaki/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
/hom

    Target=Jsc | TrainR2=0.634 | CV10R2=0.525 | OODR2=-3.755 | Overfit_suspected
  --- Target: Voc ---
    Target=Voc | TrainR2=0.671 | CV10R2=0.609 | OODR2=-2.952 | OK
  --- Target: FF ---
    Target=FF | TrainR2=0.345 | CV10R2=0.269 | OODR2=-16.672 | OK
  --- Target: PCEmax ---


...

/home/is/i-yasuaki/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/is/i-yasuaki/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/is/i-yasuaki/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/is/i-yasuaki/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
/hom

/home/is/i-yasuaki/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/is/i-yasuaki/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/is/i-yasuaki/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/is/i-yasuaki/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(
/hom

    Target=Jsc | TrainR2=0.876 | CV10R2=0.601 | OODR2=-1.589 | Overfit_suspected
  --- Target: Voc ---
    Target=Voc | TrainR2=0.882 | CV10R2=0.289 | OODR2=-15.316 | Overfit_suspected
  --- Target: FF ---
    Target=FF | TrainR2=0.687 | CV10R2=-0.850 | OODR2=-3.449 | Overfit_suspected
  --- Target: PCEmax ---
    Target=PCEmax | TrainR2=0.837 | CV10R2=0.539 | OODR2=0.382 | Overfit_suspected

=== Processing file: m_base.csv ===
  --- Target: Jsc ---


/home/is/i-yasuaki/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (2000) reached and the optimization hasn't converged yet.
  warnings.warn(


    Target=Jsc | TrainR2=0.306 | CV10R2=0.173 | OODR2=-46.733 | Overfit_suspected
  --- Target: Voc ---
    Target=Voc | TrainR2=0.324 | CV10R2=0.266 | OODR2=-128.683 | OK
  --- Target: FF ---
    Target=FF | TrainR2=0.125 | CV10R2=0.054 | OODR2=-222.149 | OK
  --- Target: PCEmax ---
    Target=PCEmax | TrainR2=0.322 | CV10R2=0.186 | OODR2=-102.034 | Overfit_suspected

=== Processing file: m_base_OH_rebuilt.csv ===
  --- Target: Jsc ---
    Target=Jsc | TrainR2=0.751 | CV10R2=0.571 | OODR2=-16.524 | Overfit_suspected
  --- Target: Voc ---
    Target=Voc | TrainR2=0.443 | CV10R2=0.110 | OODR2=-0.012 | Overfit_suspected
  --- Target: FF ---
    Target=FF | TrainR2=0.223 | CV10R2=-0.117 | OODR2=-41.329 | Overfit_suspected
  --- Target: PCEmax ---
    Target=PCEmax | TrainR2=0.726 | CV10R2=0.523 | OODR2=-14.392 | Overfit_suspected

=== Processing file: m_base_FP_rebuilt.csv ===
  --- Target: Jsc ---
    Target=Jsc | TrainR2=0.761 | CV10R2=0.641 | OODR2=-5.437 | Overfit_suspected
  --- Targ

In [5]:
##samle ID

In [6]:
import joblib
import pandas as pd
import numpy as np
import os
from datetime import datetime
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.inspection import permutation_importance

# --- Seed固定 ---
np.random.seed(42)

# --- 設定 ---
base_path = "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/rebuilt_merged_data/"

file_names = [
    "n_base.csv", "n_base_OH_rebuilt.csv", "n_base_FP_rebuilt.csv",
    "n_all.csv",  "n_all_OH_rebuilt.csv",  "n_all_FP_rebuilt.csv",
    "m_base.csv", "m_base_OH_rebuilt.csv", "m_base_FP_rebuilt.csv",
    "m_all.csv",  "m_all_OH_rebuilt.csv",  "m_all_FP_rebuilt.csv"
]

target_vars = ["Jsc", "Voc", "FF", "PCEmax"]
today = datetime.now().strftime("%Y%m%d")

# 【新条件】出力先フォルダの設定 (日付付きフォルダ)
folder_name = f"{today}_Rebuilt_12Files_Results"
output_path = os.path.join("./", folder_name)
os.makedirs(output_path, exist_ok=True)

print(f"Output Folder: {output_path}")

# --- 関数定義 ---

def create_ood_split(df, feature_cols, ood_ratio=0.1):
    X = df[feature_cols].values
    centroid = np.mean(X, axis=0)
    distances = np.linalg.norm(X - centroid, axis=1)
    num_ood = max(1, int(len(df) * ood_ratio))
    ood_indices = np.argsort(distances)[-num_ood:]
    df_train = df.drop(df.index[ood_indices])
    df_test = df.iloc[ood_indices]
    return df_train, df_test

def remove_collinear_features(df, feature_cols, threshold=0.99999):
    if len(feature_cols) < 2: return feature_cols
    X = df[feature_cols]
    corr_matrix = X.corr().abs()
    corr_matrix = corr_matrix.fillna(0)
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > threshold)]
    if len(to_drop) > 0:
        print(f"  [Cleaning] Removed {len(to_drop)} duplicated features")
    return [c for c in feature_cols if c not in to_drop]

# --- メイン処理 ---
summary_data = []
feature_importance_data = []

print(f"--- MonMLP Analysis Started: {today} ---")
print(f"Path: {base_path}")

for file in file_names:
    input_file = os.path.join(base_path, file)
    if not os.path.exists(input_file):
        print(f"Skipping: {file} (Not found)")
        continue
        
    print(f"\n=== Processing MonMLP: {file} ===")
    
    try:
        # ID列(X)をインデックスとして読み込むが、列としては使わない
        df_raw = pd.read_csv(input_file, index_col=0).drop(columns=['X'], errors='ignore')
    except Exception as e:
        print(f"Error reading {file}: {e}")
        continue
    
    for target in target_vars:
        if target not in df_raw.columns: continue
        
        # 1. データ整理
        df_temp = df_raw.select_dtypes(include=[np.number]).dropna()
        feature_cols = [c for c in df_temp.columns if c not in target_vars]
        variances = df_temp[feature_cols].var()
        feature_cols = variances[variances > 0].index.tolist()
        feature_cols = remove_collinear_features(df_temp, feature_cols)
        
        if len(df_temp) < 20 or len(feature_cols) == 0: continue

        # 2. OOD分割
        df_train, df_test = create_ood_split(df_temp, feature_cols, ood_ratio=0.1)

        # 3. スケーリング
        scaler = MinMaxScaler(feature_range=(-1, 1))
        X_train_scaled = scaler.fit_transform(df_train[feature_cols])
        y_train = df_train[target]
        X_test_scaled = scaler.transform(df_test[feature_cols])
        y_test = df_test[target]

        # 4. モデル構築 (旧条件: 高速化設定)
        mlp = MLPRegressor(
            max_iter=1000, 
            random_state=42, 
            solver='adam', 
            early_stopping=True,
            n_iter_no_change=20
        )
        
        param_grid = {
            'hidden_layer_sizes': [(64,), (128,), (64, 32)],
            'alpha': [0.001, 0.01],
            'learning_rate_init': [0.001, 0.01]
        }

        grid = GridSearchCV(
            mlp, param_grid, 
            cv=KFold(10, shuffle=True, random_state=42), 
            scoring='neg_mean_squared_error', n_jobs=-1
        )
        grid.fit(X_train_scaled, y_train)
        best_model = grid.best_estimator_
        
        # 5. 評価
        y_pred_train = best_model.predict(X_train_scaled)
        r2 = r2_score(y_train, y_pred_train)
        rmse_val = np.sqrt(mean_squared_error(y_train, y_pred_train))
        mae_val = mean_absolute_error(y_train, y_pred_train)
        rpd = np.std(y_train) / rmse_val if rmse_val != 0 else 0

        y_pred_ood = best_model.predict(X_test_scaled)
        ood_r2 = r2_score(y_test, y_pred_ood)
        ood_rmse = np.sqrt(mean_squared_error(y_test, y_pred_ood))

        summary_data.append({
            "File": file, "Target": target, 
            "R2": r2, "RMSE": rmse_val, "MAE": mae_val, "RPD": rpd,
            "n_samples": len(df_temp), "n_features": len(feature_cols),
            "OOD_R2": ood_r2, "OOD_RMSE": ood_rmse, 
            "Best_Params": str(grid.best_params_)
        })

        # 6. 変数重要度
        print(f"  Computing importance for {target}...")
        perm_importance = permutation_importance(
            best_model, X_train_scaled, y_train, 
            n_repeats=5, random_state=42, n_jobs=-1
        )
        
        indices = np.argsort(perm_importance.importances_mean)[::-1]
        for idx in indices:
            val = perm_importance.importances_mean[idx]
            if val > 1e-5: 
                feature_importance_data.append({
                    "File": file, "Target": target,
                    "Feature": feature_cols[idx],
                    "Importance_Mean": val,
                    "Importance_Std": perm_importance.importances_std[idx]
                })

        print(f"  Target: {target} | R2: {r2:.3f} | OOD_R2: {ood_r2:.3f}")

        # --- (新条件) 保存処理 ---
        # 1. モデルの保存
        model_filename = f"fixed_{today}_MonMLP_{file.replace('.csv', '')}_{target}_model.joblib"
        joblib.dump(best_model, os.path.join(output_path, model_filename))

        # 2. プロット用データの保存 (SampleIDを追加)
        df_pred = pd.DataFrame({
            'SampleID': np.concatenate([y_train.index, y_test.index]), # SampleIDを保存
            'Observed': np.concatenate([y_train.values, y_test.values]),
            'Predicted': np.concatenate([y_pred_train, y_pred_ood]),
            'Type': ['Train'] * len(y_train) + ['OOD_Test'] * len(y_test)
        })
        pred_filename = f"fixed_{today}_MonMLP_{file.replace('.csv', '')}_{target}_pred.csv"
        df_pred.to_csv(os.path.join(output_path, pred_filename), index=False)

# --- サマリー保存 ---
df_sum = pd.DataFrame(summary_data)
path_sum = os.path.join(output_path, f"fixed_{today}_MonMLP_summary.csv")
df_sum.to_csv(path_sum, index=False)

df_imp = pd.DataFrame(feature_importance_data)
path_imp = os.path.join(output_path, f"fixed_{today}_MonMLP_feature_importance.csv")
df_imp.to_csv(path_imp, index=False)

print(f"\nMonMLP Process Finished.\nSummary: {path_sum}")

Output Folder: ./20251217_Rebuilt_12Files_Results
--- MonMLP Analysis Started: 20251217 ---
Path: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/rebuilt_merged_data/

=== Processing MonMLP: n_base.csv ===
  Computing importance for Jsc...
  Target: Jsc | R2: 0.554 | OOD_R2: -0.252
  Computing importance for Voc...
  Target: Voc | R2: 0.852 | OOD_R2: -1.645
  Computing importance for FF...
  Target: FF | R2: 0.573 | OOD_R2: -1.863


/home/is/i-yasuaki/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/is/i-yasuaki/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


  Computing importance for PCEmax...
  Target: PCEmax | R2: 0.449 | OOD_R2: -0.876

=== Processing MonMLP: n_base_OH_rebuilt.csv ===
  Computing importance for Jsc...
  Target: Jsc | R2: 0.859 | OOD_R2: -0.567
  Computing importance for Voc...
  Target: Voc | R2: 0.599 | OOD_R2: 0.257
  Computing importance for FF...
  Target: FF | R2: -0.069 | OOD_R2: -0.142
  Computing importance for PCEmax...
  Target: PCEmax | R2: 0.825 | OOD_R2: -0.300

=== Processing MonMLP: n_base_FP_rebuilt.csv ===
  [Cleaning] Removed 46 duplicated features
  Computing importance for Jsc...
  Target: Jsc | R2: 0.755 | OOD_R2: 0.480
  [Cleaning] Removed 46 duplicated features
  Computing importance for Voc...
  Target: Voc | R2: 0.855 | OOD_R2: 0.582
  [Cleaning] Removed 46 duplicated features
  Computing importance for FF...
  Target: FF | R2: 0.575 | OOD_R2: -1.620
  [Cleaning] Removed 46 duplicated features
  Computing importance for PCEmax...
  Target: PCEmax | R2: 0.673 | OOD_R2: 0.363

=== Processing MonM

/home/is/i-yasuaki/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/is/i-yasuaki/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/is/i-yasuaki/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/is/i-yasuaki/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/hom

  Computing importance for Jsc...
  Target: Jsc | R2: 0.781 | OOD_R2: -5.082
  [Cleaning] Removed 9 duplicated features


...

  Computing importance for Jsc...
  Target: Jsc | R2: 0.700 | OOD_R2: 0.245
  [Cleaning] Removed 46 duplicated features
  Computing importance for Voc...
  Target: Voc | R2: 0.727 | OOD_R2: -21.613
  [Cleaning] Removed 46 duplicated features
  Computing importance for FF...
  Target: FF | R2: 0.592 | OOD_R2: -1.059
  [Cleaning] Removed 46 duplicated features
  Computing importance for PCEmax...
  Target: PCEmax | R2: 0.871 | OOD_R2: -0.277

=== Processing MonMLP: m_base.csv ===


/home/is/i-yasuaki/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/is/i-yasuaki/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/is/i-yasuaki/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/is/i-yasuaki/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/hom

  Computing importance for Jsc...
  Target: Jsc | R2: 0.174 | OOD_R2: -35.028
  Computing importance for Voc...
  Target: Voc | R2: 0.348 | OOD_R2: -98.744
  Computing importance for FF...
  Target: FF | R2: 0.207 | OOD_R2: -184.812


/home/is/i-yasuaki/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/home/is/i-yasuaki/miniconda3/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


  Computing importance for PCEmax...
  Target: PCEmax | R2: 0.464 | OOD_R2: -155.546

=== Processing MonMLP: m_base_OH_rebuilt.csv ===
  Computing importance for Jsc...
  Target: Jsc | R2: 0.805 | OOD_R2: -16.080
  Computing importance for Voc...
  Target: Voc | R2: 0.668 | OOD_R2: -1.507
  Computing importance for FF...
  Target: FF | R2: 0.651 | OOD_R2: -11.235
  Computing importance for PCEmax...
  Target: PCEmax | R2: 0.745 | OOD_R2: -13.759

=== Processing MonMLP: m_base_FP_rebuilt.csv ===
  [Cleaning] Removed 40 duplicated features
  Computing importance for Jsc...
  Target: Jsc | R2: 0.723 | OOD_R2: -4.385
  [Cleaning] Removed 40 duplicated features
  Computing importance for Voc...
  Target: Voc | R2: 0.773 | OOD_R2: -18.795
  [Cleaning] Removed 40 duplicated features
  Computing importance for FF...
  Target: FF | R2: 0.504 | OOD_R2: -5.271
  [Cleaning] Removed 40 duplicated features
  Computing importance for PCEmax...
  Target: PCEmax | R2: 0.710 | OOD_R2: -4.795

=== Proces